<a href="https://colab.research.google.com/github/ValentinaZubareva2906/make_AI_product/blob/main/chain/3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-classic openai langchain-openai langchain-community -U -q

In [2]:
import re
import pandas as pd
from tqdm import tqdm
from getpass import getpass

from langchain_classic import PromptTemplate
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

In [3]:
from langchain_classic.schema.output_parser import StrOutputParser

## Если используете ключ из курса, запустите эти ячейки 👇


In [4]:
from langchain_openai import ChatOpenAI
from getpass import getpass

course_api_key= "sk-CTWDfcT-MqN2gUhZh_3qbA"

# инициализируем языковую модель
llm = ChatOpenAI(api_key=course_api_key, model='gpt-4o-mini',
                 base_url="https://aleron-llm.neuraldeep.tech/")

## Задание 3.2.9 🤔 Кажется, это что-то на LLM-ском? 🧐

In [5]:
df = pd.read_csv("https://stepik.org/media/attachments/lesson/1110883/raw_texts.csv")
df.head()

,raw_text
0,"The sun was setting, casting long shadows over..."
1,"Le soleil se couchait, jetant de longues ombre..."
2,"El sol se estaba poniendo, proyectando largas ..."
3,"La ciudad estaba llena de vida, sus calles lle..."
4,"La ville était pleine de vie, ses rues remplie..."


Напишем функцию, которая очистит текст от ненужных символов: `¿, ¡, £`

In [6]:
def clean_text(inputs: dict) -> dict:
    inputs["text"] = re.sub(r'[¿¡£]+', '', inputs["text"])
    return inputs

Будем просить у модели определять язык и имя главного персонажа и выдавать ответ в виде словаря. Для этого создадим `Output parser`, с которым вы уже познакомились в прошлых уроках. НЕ ОБЯЗАТЕЛЬНЫЙ ШАГ

In [7]:
# Определим схемы ответа
language_schema = ResponseSchema(
    name="language",
    description=("Извлеки язык, на котором был написан данный текст."
        "Пример: 'Русский' → 'Russian'."
        "Пример: 'Deutsch' → 'German'."
    )
)


person_schema = ResponseSchema(
    name="person",
    description=("Извлеки имя главного героя из приведенного текста."
        "Имя героя должно быть на том языке, на котором написан {text}"
    )
)

response_schemas = [language_schema, person_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas) # Создаём парсер и подаём в него список со схемами
format_instructions = output_parser.get_format_instructions() # Получаем инструкции по форматированию ответа

Напишем шаблон промпта со своим вопросом и инструкциями по форматированию ответа. Будем передавать в этот промпт сырой текст

In [8]:
template = """\
Из {text} извлеки информацию в строго указанном формате.

Поля для извлечения:
- language: название языка на английском языке.
- person: имя персонажа на языке, на котором написан предсавленный текст.

Текст:
{text}

Инструкции по формату:
{format_instructions}
"""


In [9]:
prompt = PromptTemplate(input_variables=['text', 'format_instructions'], template=template)

Создадим цепочку с помощью `LCEL`

In [10]:
chain = (
    clean_text
    | prompt.partial(format_instructions=format_instructions)
    | llm
    | output_parser
)

In [11]:
df_new = []

In [12]:
for text in tqdm(df['raw_text']):
    try:
        #df_new.append(clean_text(text))
        result = chain.invoke({'text': text})
        df_new.append(result)

    except Exception as e:
        print(f"Ошибка: {e}")
        df_new.append(None)

 15%|█▌        | 2/13 [00:03<00:18,  1.67s/it]

Ошибка: Error code: 400 - {'error': {'message': 'litellm.BadRequestError: OpenAIException - The OpenAI account associated with this API key has been deactivated. If you are the developer for this OpenAI app, please check your email for more information. If you are seeing this error while using another app or site, please reach out to them for more help.. Received Model Group=gpt-4o-mini\nAvailable Model Group Fallbacks=None', 'type': 'invalid_request_error', 'param': None, 'code': '400'}}


 69%|██████▉   | 9/13 [00:12<00:04,  1.17s/it]

Ошибка: Error code: 400 - {'error': {'message': 'litellm.BadRequestError: OpenAIException - The OpenAI account associated with this API key has been deactivated. If you are the developer for this OpenAI app, please check your email for more information. If you are seeing this error while using another app or site, please reach out to them for more help.. Received Model Group=gpt-4o-mini\nAvailable Model Group Fallbacks=None', 'type': 'invalid_request_error', 'param': None, 'code': '400'}}


100%|██████████| 13/13 [00:18<00:00,  1.43s/it]


In [13]:
for i, item in enumerate(df_new):
    if item is None:
        df_new[i] = {
            "language": None,
            "person": None
        }

In [14]:
df_new

[{'language': 'English', 'person': 'John'},
 {'language': None, 'person': None},
 {'language': 'Spanish', 'person': 'Carlos'},
 {'language': 'Spanish', 'person': 'Juan'},
 {'language': 'French', 'person': 'Jean'},
 {'language': 'German', 'person': 'Johann'},
 {'language': 'German', 'person': 'Hans'},
 {'language': 'Russian', 'person': 'Анна'},
 {'language': None, 'person': None},
 {'language': 'Spanish', 'person': 'Maria'},
 {'language': 'French', 'person': 'Sophie'},
 {'language': 'Russian', 'person': 'Иван'},
 {'language': 'Italian', 'person': 'Giovanni'}]

In [15]:
# 1. Превращаем dict_list в DataFrame
parsed_df = pd.DataFrame(df_new)

# 2. Объединяем с исходным df
df = pd.concat([df, parsed_df], axis=1)

# Сохраняем или выводим
print(df.head())

                                            raw_text language  person
0  The sun was setting, casting long shadows over...  English    John
1  Le soleil se couchait, jetant de longues ombre...     None    None
2  El sol se estaba poniendo, proyectando largas ...  Spanish  Carlos
3  La ciudad estaba llena de vida, sus calles lle...  Spanish    Juan
4  La ville était pleine de vie, ses rues remplie...   French    Jean


In [16]:
df.columns = ['text', 'language', 'main_character']

In [18]:
def process_text(text):
    try:
        result = clean_text({"text": text})
        return result["text"]
    except:
        return None

df['text'] = df['text'].apply(process_text)


Сохраним всё в итоговый файл. Убедитесь, что на этом этапе у вас в столбцах

- `text` - очищенный текст (без символов ¿, ¡, £)
- `language` - язык, на котором написан текст (название языка указать на английском языке)
- `main_character` - имя главного персонажа в тексте (указать на том языке, на котором и написан сам текст)

In [19]:
df[['text', 'language', 'main_character']].to_csv('3.2.9_solution.csv', index=False)